# Fruit Classification Feature Extraction

This notebook compares two segmentation approaches:

1. Otsu Thresholding
2. GrabCut

For each segmented fruit, we extract:

- Shape features
- Hu Moments
- Color features
- Texture features (GLCM)

Two datasets are generated:

- X_otsu.csv / y_otsu.csv
- X_grabcut.csv / y_grabcut.csv

These datasets will later be used to train separate machine learning models and compare the impact of segmentation quality on classification performance.

In [13]:
import cv2
import numpy as np
import pandas as pd
import os
import glob

from skimage.feature import graycomatrix, graycoprops

## Utility Functions

Functions for:

- image loading
- illumination normalization (CLAHE)
- connected component filtering

In [14]:
def load_image(filepath):

    img = cv2.imread(filepath)

    if img is None:
        return None

    return cv2.cvtColor(
        img,
        cv2.COLOR_BGR2RGB
    )


def normalize_lighting(img_rgb):

    lab = cv2.cvtColor(
        img_rgb,
        cv2.COLOR_RGB2LAB
    )

    l, a, b = cv2.split(lab)

    clahe = cv2.createCLAHE(
        clipLimit=3.0,
        tileGridSize=(8, 8)
    )

    l = clahe.apply(l)

    lab = cv2.merge([l, a, b])

    return cv2.cvtColor(
        lab,
        cv2.COLOR_LAB2RGB
    )


def keep_largest_component(mask):

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask)

    if num_labels <= 1:
        return mask

    largest = (
        1 +
        np.argmax(
            stats[1:, cv2.CC_STAT_AREA]
        )
    )

    return np.uint8(labels == largest) * 255

## Segmentation Methods

### Otsu

Required baseline method.

### GrabCut

Improved segmentation method used for comparison.

In [15]:
def segment_otsu(img_rgb):

    gray = cv2.cvtColor(
        img_rgb,
        cv2.COLOR_RGB2GRAY
    )

    gray = cv2.GaussianBlur(
        gray,
        (7, 7),
        0
    )

    _, mask = cv2.threshold(
        gray,
        0,
        255,
        cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
    )

    kernel = np.ones((5, 5), np.uint8)

    mask = cv2.morphologyEx(
        mask,
        cv2.MORPH_CLOSE,
        kernel
    )

    mask = keep_largest_component(mask)

    return mask

In [16]:
def segment_grabcut(img_rgb):

    h, w = img_rgb.shape[:2]

    mask = np.zeros(
        (h, w),
        np.uint8
    )

    rect = (
        int(w * 0.15),
        int(h * 0.15),
        int(w * 0.70),
        int(h * 0.70)
    )

    bgdModel = np.zeros((1, 65), np.float64)
    fgdModel = np.zeros((1, 65), np.float64)

    cv2.grabCut(
        img_rgb,
        mask,
        rect,
        bgdModel,
        fgdModel,
        5,
        cv2.GC_INIT_WITH_RECT
    )

    mask = np.where(
        (mask == cv2.GC_FGD) |
        (mask == cv2.GC_PR_FGD),
        255,
        0
    ).astype(np.uint8)

    mask = keep_largest_component(mask)

    kernel = np.ones((7, 7), np.uint8)

    mask = cv2.morphologyEx(
        mask,
        cv2.MORPH_CLOSE,
        kernel
    )

    return mask

## Feature Extraction

Extracted features:

### Shape

- Area
- Perimeter
- Circularity
- Aspect Ratio
- Extent
- Solidity

### Hu Moments

7 invariant moments.

### Color

Mean RGB values.

### Texture

GLCM:
- Contrast
- Correlation
- Energy
- Homogeneity

In [17]:
def extract_features(img_rgb, mask):

    features = {}

    contours, _ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    if not contours:
        return None

    c = max(
        contours,
        key=cv2.contourArea
    )

    area = cv2.contourArea(c)

    perimeter = cv2.arcLength(
        c,
        True
    )

    features["area"] = area

    features["perimeter"] = perimeter

    features["circularity"] = (
        (4 * np.pi * area) / (perimeter ** 2)
        if perimeter > 0 else 0
    )

    x, y, w, h = cv2.boundingRect(c)

    features["aspect_ratio"] = (
        w / h
        if h > 0 else 0
    )

    features["extent"] = (
        area / (w * h)
        if w * h > 0 else 0
    )

    hull = cv2.convexHull(c)

    hull_area = cv2.contourArea(hull)

    features["solidity"] = (
        area / hull_area
        if hull_area > 0 else 0
    )

    moments = cv2.moments(c)

    hu = cv2.HuMoments(
        moments
    ).flatten()

    for i, value in enumerate(hu):

        features[f"hu_moment_{i}"] = (
            -np.sign(value)
            * np.log10(np.abs(value))
            if value != 0
            else 0
        )

    mean_rgb = cv2.mean(
        img_rgb,
        mask=mask
    )

    features["mean_R"] = mean_rgb[0]
    features["mean_G"] = mean_rgb[1]
    features["mean_B"] = mean_rgb[2]

    gray = cv2.cvtColor(
        img_rgb,
        cv2.COLOR_RGB2GRAY
    )

    pixels = gray[mask > 0]

    if len(pixels) > 100:

        side = int(np.sqrt(len(pixels)))

        pixels = pixels[: side * side]

        texture_img = pixels.reshape(
            side,
            side
        )

        glcm = graycomatrix(
            texture_img,
            distances=[1],
            angles=[0],
            levels=256,
            symmetric=True,
            normed=True
        )

        features["glcm_contrast"] = graycoprops(
            glcm,
            "contrast"
        )[0, 0]

        features["glcm_correlation"] = graycoprops(
            glcm,
            "correlation"
        )[0, 0]

        features["glcm_energy"] = graycoprops(
            glcm,
            "energy"
        )[0, 0]

        features["glcm_homogeneity"] = graycoprops(
            glcm,
            "homogeneity"
        )[0, 0]

    else:

        features["glcm_contrast"] = 0
        features["glcm_correlation"] = 0
        features["glcm_energy"] = 0
        features["glcm_homogeneity"] = 0

    return features

## Dataset Configuration

In [18]:
diretorio_base = "../dataset"

limite_por_classe = 400

pastas_alvo = [

    ("good_quality_fruits/Apple_Good", "fresh"),
    ("bad_quality_fruits/Apple_Bad", "rotten"),
]

dados_otsu = []
dados_grabcut = []

## Feature Extraction Loop

In [19]:
for pasta_relativa, label_classe in pastas_alvo:

    caminho_completo = os.path.join(
        diretorio_base,
        pasta_relativa
    )

    arquivos = glob.glob(
        os.path.join(
            caminho_completo,
            "*.[jJ][pP]*"
        )
    )

    contador = 0

    print(
        f"Processing {pasta_relativa}"
    )

    for arquivo in arquivos:

        if contador >= limite_por_classe:
            break

        img = load_image(arquivo)

        if img is None:
            continue

        img_norm = normalize_lighting(img)

        # OTSU

        mask_otsu = segment_otsu(
            img_norm
        )

        feat_otsu = extract_features(
            img_norm,
            mask_otsu
        )

        if feat_otsu is not None:

            feat_otsu["label"] = label_classe

            dados_otsu.append(
                feat_otsu
            )

        # GRABCUT

        mask_gc = segment_grabcut(
            img_norm
        )

        feat_gc = extract_features(
            img_norm,
            mask_gc
        )

        if feat_gc is not None:

            feat_gc["label"] = label_classe

            dados_grabcut.append(
                feat_gc
            )

        contador += 1

Processing good_quality_fruits/Apple_Good


Processing bad_quality_fruits/Apple_Bad


## Save Feature Tables

In [20]:
os.makedirs(
    "../outputs",
    exist_ok=True
)

# OTSU

df_otsu = pd.DataFrame(
    dados_otsu
)

X_otsu = df_otsu.drop(
    columns=["label"]
)

y_otsu = df_otsu["label"]

X_otsu.to_csv(
    "../outputs/X_otsu.csv",
    index=False
)

y_otsu.to_csv(
    "../outputs/y_otsu.csv",
    index=False
)

# GRABCUT

df_gc = pd.DataFrame(
    dados_grabcut
)

X_gc = df_gc.drop(
    columns=["label"]
)

y_gc = df_gc["label"]

X_gc.to_csv(
    "../outputs/X_grabcut.csv",
    index=False
)

y_gc.to_csv(
    "../outputs/y_grabcut.csv",
    index=False
)

print()
print("Done.")
print("Otsu samples:", len(df_otsu))
print("GrabCut samples:", len(df_gc))


Done.
Otsu samples: 800
GrabCut samples: 800


## Next Step

Train two independent classifiers:

### Model A

Input:
- X_otsu.csv
- y_otsu.csv

### Model B

Input:
- X_grabcut.csv
- y_grabcut.csv

Suggested classifiers:

- Random Forest
- SVM
- XGBoost

Compare:

- Accuracy
- Precision
- Recall
- F1-score

This will directly measure the impact of segmentation quality on classification performance.